# Proyecto Final — Aprendizaje No Supervisado  
## Análisis de salud mental y estilo de vida mediante clustering

**Dataset público:** `dhrubangtalukdar/global-mental-health-and-lifestyle-survey-dataset`  
**Fuente:** Kaggle  
**Módulo:** Aprendizaje No Supervisado  
**Técnicas principales:** Gaussian Mixture Models, Spectral Clustering, HDBSCAN, Isolation Forest y UMAP.

---

### Relación con la guía
Este notebook está organizado siguiendo las partes pedidas en la guía del proyecto final:

1. Introducción y planteamiento del problema.  
2. Objetivos.  
3. Dataset público, fuente, observaciones y variables.  
4. Análisis exploratorio de datos, calidad de datos, estadísticas, distribuciones, correlaciones y outliers.  
5. Preprocesamiento con justificación.  
6. Metodología de clustering con al menos dos algoritmos distintos a K-Means.  
7. Optimización de hiperparámetros.  
8. Evaluación con métricas internas y externas cuando existen etiquetas.  
9. Visualización con UMAP, técnica distinta de PCA.  
10. Discusión, conclusiones y referencias.

## 1. Introducción

La salud mental está influenciada por múltiples factores relacionados con el estilo de vida, como el sueño, el estrés laboral, la actividad física, el tiempo de pantalla, el apoyo social y la satisfacción personal. En este proyecto se utilizan técnicas de aprendizaje no supervisado para identificar grupos de personas con patrones similares de comportamiento y bienestar psicológico.

El objetivo no es diagnosticar enfermedades, sino explorar si existen perfiles o agrupamientos naturales dentro de los datos. Este enfoque puede ser útil para comprender tendencias generales y detectar perfiles de riesgo, siempre considerando que el análisis es exploratorio y depende de la calidad del dataset.

## 2. Objetivos

### Objetivo general
Aplicar técnicas de aprendizaje no supervisado sobre un dataset público de salud mental y estilo de vida, realizando un flujo completo de análisis desde el EDA hasta la interpretación crítica de los resultados.

### Objetivos específicos
- Describir el dataset, sus variables y su fuente pública.
- Analizar la calidad de los datos, valores faltantes, distribuciones, correlaciones y posibles outliers.
- Preprocesar las variables mediante codificación, imputación y escalado.
- Aplicar y comparar al menos dos algoritmos de clustering distintos a K-Means.
- Optimizar hiperparámetros relevantes de los modelos utilizados.
- Evaluar los agrupamientos con métricas internas y, cuando sea posible, métricas externas.
- Visualizar la estructura de los datos mediante UMAP, una técnica diferente de PCA.
- Interpretar críticamente los perfiles encontrados.

## 3. Instalación, librerías y configuración

Esta celda instala dependencias que pueden no estar disponibles en un entorno limpio. Si ya están instaladas, puede comentarse la línea de instalación.

In [ ]:
# ============================================================
# INSTALACIÓN DE DEPENDENCIAS
# ============================================================
# En Google Colab o un entorno nuevo, descomentar y ejecutar:
# !pip install kagglehub hdbscan umap-learn scikit-learn pandas matplotlib seaborn

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import PolynomialFeatures

# Librerías avanzadas usadas para visualización y clustering basado en densidad.
# Si no están instaladas, revisar la celda de instalación anterior.
try:
    import umap.umap_ as umap
except Exception as e:
    raise ImportError("Falta instalar umap-learn. Ejecutar: pip install umap-learn") from e

try:
    import hdbscan
except Exception as e:
    raise ImportError("Falta instalar hdbscan. Ejecutar: pip install hdbscan") from e

# Semilla para reproducibilidad.
SEED = 42
np.random.seed(SEED)

# Configuración visual general.
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.figsize"] = (10, 5)

print("Librerías cargadas correctamente.")

## 4. Carga del dataset público

El dataset se descarga desde Kaggle usando `kagglehub`. Si se ejecuta en un entorno sin acceso a Kaggle, se puede descargar manualmente el archivo `mental_health.csv` desde Kaggle y colocarlo en la misma carpeta del notebook.

In [ ]:
# ============================================================
# CARGA DEL DATASET
# ============================================================

DATASET_REF = "dhrubangtalukdar/global-mental-health-and-lifestyle-survey-dataset"
LOCAL_CSV = "mental_health.csv"

try:
    # Opción 1: si el archivo ya está en la carpeta del notebook.
    if os.path.exists(LOCAL_CSV):
        df = pd.read_csv(LOCAL_CSV)
        print(f"Dataset cargado desde archivo local: {LOCAL_CSV}")
    else:
        # Opción 2: descarga directa desde Kaggle.
        import kagglehub
        path = kagglehub.dataset_download(DATASET_REF)
        csv_path = os.path.join(path, "mental_health.csv")
        df = pd.read_csv(csv_path)
        print(f"Dataset descargado desde Kaggle: {csv_path}")
except Exception as e:
    raise RuntimeError(
        "No se pudo cargar el dataset. "
        "Descargar manualmente mental_health.csv desde Kaggle y colocarlo junto al notebook."
    ) from e

print("Dimensión del dataset:", df.shape)
df.head()

## 5. Descripción del dataset

En esta sección se reporta la cantidad de observaciones, variables y tipos de datos. Esta parte es importante porque la guía solicita describir el dataset público, su disponibilidad y sus principales características.

In [ ]:
# ============================================================
# DESCRIPCIÓN GENERAL DEL DATASET
# ============================================================

print("Cantidad de observaciones:", df.shape[0])
print("Cantidad de variables:", df.shape[1])

resumen_variables = pd.DataFrame({
    "variable": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "valores_no_nulos": df.notna().sum().values,
    "valores_faltantes": df.isna().sum().values,
    "valores_unicos": df.nunique().values,
})

resumen_variables

In [ ]:
# ============================================================
# SEPARACIÓN DE VARIABLES NUMÉRICAS Y CATEGÓRICAS
# ============================================================

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Variables numéricas:")
print(num_cols)

print("\nVariables categóricas:")
print(cat_cols)

## 6. Análisis Exploratorio de Datos — EDA

La guía solicita analizar calidad de datos, estadísticas descriptivas básicas, distribuciones, correlaciones y posibles outliers. Por eso, el EDA se divide en varias subsecciones.

### 6.1 Calidad de datos: valores faltantes y duplicados

In [ ]:
# ============================================================
# CALIDAD DE DATOS
# ============================================================

missing_table = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje_faltante": (df.isna().mean() * 100).round(2),
}).sort_values("porcentaje_faltante", ascending=False)

print("Filas duplicadas:", df.duplicated().sum())
missing_table.head(20)

### 6.2 Estadísticas descriptivas

In [ ]:
# ============================================================
# ESTADÍSTICAS DESCRIPTIVAS DE VARIABLES NUMÉRICAS
# ============================================================

df[num_cols].describe().T.round(2)

In [ ]:
# ============================================================
# FRECUENCIAS DE VARIABLES CATEGÓRICAS
# ============================================================

for col in cat_cols:
    print("=" * 70)
    print(f"Variable: {col}")
    print(df[col].value_counts(dropna=False).head(10))

### 6.3 Visualización de distribuciones numéricas

In [ ]:
# ============================================================
# HISTOGRAMAS DE VARIABLES NUMÉRICAS
# ============================================================

cols_to_plot = num_cols[:12]

n_cols = 3
n_rows = int(np.ceil(len(cols_to_plot) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, cols_to_plot):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col.replace("_", " "))
    ax.set_xlabel("")

for ax in axes[len(cols_to_plot):]:
    ax.axis("off")

plt.suptitle("Distribuciones de variables numéricas", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 6.4 Distribuciones categóricas

In [ ]:
# ============================================================
# GRÁFICOS DE BARRAS PARA VARIABLES CATEGÓRICAS
# ============================================================

cols_to_plot = cat_cols[:8]

n_cols = 2
n_rows = int(np.ceil(len(cols_to_plot) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, cols_to_plot):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax)
    ax.set_title(col.replace("_", " "))
    ax.set_xlabel("Frecuencia")
    ax.set_ylabel("")

for ax in axes[len(cols_to_plot):]:
    ax.axis("off")

plt.suptitle("Distribuciones de variables categóricas", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 6.5 Correlaciones

In [ ]:
# ============================================================
# MATRIZ DE CORRELACIÓN
# ============================================================

corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlación de Pearson entre variables numéricas")
plt.tight_layout()
plt.show()

# Pares de variables con mayor correlación absoluta.
corr_pairs = (
    corr.where(~mask)
    .stack()
    .reset_index()
    .rename(columns={"level_0": "variable_1", "level_1": "variable_2", 0: "correlacion"})
)
corr_pairs["abs_correlacion"] = corr_pairs["correlacion"].abs()
corr_pairs = corr_pairs.sort_values("abs_correlacion", ascending=False)

corr_pairs.head(10)

### 6.6 Detección preliminar de outliers

In [ ]:
# ============================================================
# OUTLIERS MEDIANTE RANGO INTERCUARTÍLICO, IQR
# ============================================================

outlier_summary = []

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append({
        "variable": col,
        "limite_inferior": round(lower, 2),
        "limite_superior": round(upper, 2),
        "posibles_outliers": int(n_out),
        "porcentaje": round(100 * n_out / len(df), 2),
    })

outlier_summary = pd.DataFrame(outlier_summary).sort_values("posibles_outliers", ascending=False)
outlier_summary

In [ ]:
# ============================================================
# BOXPLOTS PARA VISUALIZAR POSIBLES OUTLIERS
# ============================================================

cols_to_plot = num_cols[:12]

n_cols = 3
n_rows = int(np.ceil(len(cols_to_plot) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, cols_to_plot):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col.replace("_", " "))

for ax in axes[len(cols_to_plot):]:
    ax.axis("off")

plt.suptitle("Boxplots para detección visual de outliers", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 6.7 Conclusiones previas al clustering

A partir del EDA se deben tener en cuenta los siguientes puntos antes de aplicar clustering:

- Las variables se encuentran en escalas distintas, por lo que es necesario aplicar escalado.
- Existen variables categóricas que deben codificarse antes del modelado.
- Los posibles outliers pueden afectar modelos sensibles a distancia, por lo que se incluye Isolation Forest como análisis complementario.
- Si existe una variable como `Has_Mental_Health_Issue`, no se usará para entrenar los clusters, sino solo como referencia externa para evaluar si los agrupamientos tienen alguna relación con esa etiqueta.

## 7. Preprocesamiento

El preprocesamiento incluye imputación simple, codificación de variables categóricas y normalización. Esto se justifica porque los algoritmos de clustering basados en distancia o densidad son sensibles a la escala y al tipo de representación de las variables.

In [ ]:
# ============================================================
# DEFINICIÓN DE VARIABLE ETIQUETA, SOLO PARA EVALUACIÓN EXTERNA
# ============================================================

# Esta variable no se usará para entrenar los modelos.
# Se guarda solamente para métricas externas, si existe en el dataset.
target_col = "Has_Mental_Health_Issue" if "Has_Mental_Health_Issue" in df.columns else None
print("Variable de referencia externa:", target_col)

In [ ]:
# ============================================================
# IMPUTACIÓN SIMPLE
# ============================================================

# Copia para no modificar el dataset original.
df_proc = df.copy()

# Imputación numérica: se usa la mediana porque es robusta ante outliers.
for col in num_cols:
    df_proc[col] = df_proc[col].fillna(df_proc[col].median())

# Imputación categórica: se usa la moda o la categoría "Desconocido" si no hay moda.
for col in cat_cols:
    if df_proc[col].mode(dropna=True).empty:
        df_proc[col] = df_proc[col].fillna("Desconocido")
    else:
        df_proc[col] = df_proc[col].fillna(df_proc[col].mode(dropna=True)[0])

print("Valores faltantes luego de imputar:", int(df_proc.isna().sum().sum()))

In [ ]:
# ============================================================
# CODIFICACIÓN DE VARIABLES CATEGÓRICAS
# ============================================================

# La etiqueta externa no debe entrar al clustering.
cols_modelo = df_proc.columns.tolist()
if target_col in cols_modelo:
    cols_modelo.remove(target_col)

X_df = df_proc[cols_modelo].copy()

# One-hot encoding para variables categóricas nominales.
X_df = pd.get_dummies(X_df, drop_first=False)

# Conversión explícita a float para evitar problemas con tipos booleanos.
X_raw = X_df.astype(float).values

print("Matriz antes del escalado:", X_raw.shape)
print("Cantidad de variables finales:", X_df.shape[1])
X_df.head()

In [ ]:
# ============================================================
# ESCALADO DE VARIABLES
# ============================================================

# StandardScaler centra las variables en media 0 y desviación estándar 1.
# Esto es fundamental para clustering porque evita que variables con valores grandes dominen las distancias.
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print("Matriz final para clustering:", X.shape)

## 8. Visualización no supervisada con UMAP

La guía solicita usar al menos una técnica distinta de PCA para visualizar los datos. Se utiliza UMAP porque conserva estructura local y puede revelar agrupamientos no lineales.

In [ ]:
# ============================================================
# REDUCCIÓN DE DIMENSIONALIDAD CON UMAP
# ============================================================

reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.10,
    metric="euclidean",
    random_state=SEED,
)

X_umap = reducer_2d.fit_transform(X)

plt.figure(figsize=(8, 6))
plt.scatter(X_umap[:, 0], X_umap[:, 1], s=8, alpha=0.7)
plt.title("Proyección UMAP 2D del dataset")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.tight_layout()
plt.show()

print("Dimensión de la proyección UMAP:", X_umap.shape)

## 9. Funciones auxiliares de evaluación

Se definen funciones para calcular métricas internas y externas de forma ordenada. Las métricas internas evalúan la separación y compacidad de los clusters sin usar etiquetas reales. Las métricas externas se calculan solo si existe una etiqueta de referencia.

In [ ]:
# ============================================================
# FUNCIONES DE EVALUACIÓN
# ============================================================

def contar_clusters(labels):
    # Cuenta clusters ignorando el ruido etiquetado como -1.
    labels_unicos = set(labels)
    return len(labels_unicos) - (1 if -1 in labels_unicos else 0)


def metricas_internas(X_eval, labels):
    # Calcula métricas internas si el resultado tiene al menos dos clusters válidos.
    labels = np.asarray(labels)
    mask = labels != -1
    labels_validos = labels[mask]
    X_validos = X_eval[mask]
    n_clusters = len(set(labels_validos))
    
    if n_clusters < 2 or len(labels_validos) <= n_clusters:
        return {
            "silhouette": np.nan,
            "davies_bouldin": np.nan,
            "calinski_harabasz": np.nan,
        }
    
    return {
        "silhouette": silhouette_score(X_validos, labels_validos),
        "davies_bouldin": davies_bouldin_score(X_validos, labels_validos),
        "calinski_harabasz": calinski_harabasz_score(X_validos, labels_validos),
    }


def metricas_externas(labels):
    # Calcula ARI y NMI si existe una etiqueta externa disponible.
    if target_col is None:
        return {"ari": np.nan, "nmi": np.nan}
    
    y_true = df_proc[target_col].astype(str).values
    labels = np.asarray(labels)
    mask = labels != -1
    
    if len(set(labels[mask])) < 2:
        return {"ari": np.nan, "nmi": np.nan}
    
    return {
        "ari": adjusted_rand_score(y_true[mask], labels[mask]),
        "nmi": normalized_mutual_info_score(y_true[mask], labels[mask]),
    }


def evaluar_modelo(nombre, labels, X_eval=X_umap):
    # Devuelve una fila resumida con métricas internas, externas y cantidad de clusters.
    internas = metricas_internas(X_eval, labels)
    externas = metricas_externas(labels)
    labels = np.asarray(labels)
    return {
        "modelo": nombre,
        "clusters": contar_clusters(labels),
        "ruido_%": round(100 * np.mean(labels == -1), 2),
        "silhouette": internas["silhouette"],
        "davies_bouldin": internas["davies_bouldin"],
        "calinski_harabasz": internas["calinski_harabasz"],
        "ari": externas["ari"],
        "nmi": externas["nmi"],
    }

## 10. Metodología de clustering

Se aplican tres algoritmos de clustering. La guía indica que K-Means puede usarse solo como referencia y no cuenta como uno de los dos algoritmos requeridos; por eso aquí se usan modelos distintos a K-Means:

1. **Gaussian Mixture Model, GMM:** modelo probabilístico que permite asignación suave de pertenencia a clusters.  
2. **Spectral Clustering:** método basado en grafos, útil para estructuras no lineales.  
3. **HDBSCAN:** clustering basado en densidad, capaz de detectar ruido y clusters de formas irregulares.

Además, se usa **Isolation Forest** como análisis complementario de anomalías, no como clustering principal.

## 11. Algoritmo 1 — Gaussian Mixture Model, GMM

In [ ]:
# ============================================================
# OPTIMIZACIÓN DE HIPERPARÁMETROS PARA GMM
# ============================================================

# Se prueba distinto número de componentes y tipos de covarianza.
# La elección final se basa en BIC: menor BIC indica mejor balance entre ajuste y complejidad.

gmm_resultados = []

param_grid_gmm = {
    "n_components": range(2, 11),
    "covariance_type": ["full", "tied", "diag", "spherical"],
}

for params in ParameterGrid(param_grid_gmm):
    model = GaussianMixture(
        n_components=params["n_components"],
        covariance_type=params["covariance_type"],
        random_state=SEED,
        n_init=5,
    )
    model.fit(X_umap)
    labels_tmp = model.predict(X_umap)
    m = metricas_internas(X_umap, labels_tmp)
    gmm_resultados.append({
        **params,
        "bic": model.bic(X_umap),
        "aic": model.aic(X_umap),
        "silhouette": m["silhouette"],
        "davies_bouldin": m["davies_bouldin"],
    })

gmm_resultados = pd.DataFrame(gmm_resultados).sort_values("bic")
gmm_resultados.head(10)

In [ ]:
# ============================================================
# ENTRENAMIENTO FINAL DE GMM
# ============================================================

best_gmm = gmm_resultados.iloc[0]

modelo_gmm = GaussianMixture(
    n_components=int(best_gmm["n_components"]),
    covariance_type=best_gmm["covariance_type"],
    random_state=SEED,
    n_init=10,
)

modelo_gmm.fit(X_umap)
labels_gmm = modelo_gmm.predict(X_umap)
proba_gmm = modelo_gmm.predict_proba(X_umap)

print("Mejores hiperparámetros GMM:")
print(best_gmm[["n_components", "covariance_type", "bic", "aic"]])

print("\nMétricas GMM:")
print(evaluar_modelo("GMM", labels_gmm))

In [ ]:
# ============================================================
# VISUALIZACIÓN DE GMM
# ============================================================

entropy_gmm = -(proba_gmm * np.log(proba_gmm + 1e-10)).sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc1 = axes[0].scatter(X_umap[:, 0], X_umap[:, 1], c=labels_gmm, s=8, alpha=0.7, cmap="tab10")
axes[0].set_title("GMM: clusters sobre UMAP")
axes[0].set_xlabel("UMAP 1")
axes[0].set_ylabel("UMAP 2")
plt.colorbar(sc1, ax=axes[0], label="Cluster")

sc2 = axes[1].scatter(X_umap[:, 0], X_umap[:, 1], c=entropy_gmm, s=8, alpha=0.7, cmap="viridis")
axes[1].set_title("GMM: incertidumbre de asignación")
axes[1].set_xlabel("UMAP 1")
axes[1].set_ylabel("UMAP 2")
plt.colorbar(sc2, ax=axes[1], label="Entropía")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PERFIL DE CLUSTERS GMM
# ============================================================

df_gmm = df_proc.copy()
df_gmm["cluster_gmm"] = labels_gmm

perfil_cols = [c for c in [
    "Age", "Sleep_Hours_Night", "Social_Media_Hours_Day", "Screen_Time_Hours_Day",
    "Work_Stress_Level", "Anxious_Nervous", "Feeling_Sad_Down",
    "Work_Life_Balance", "Job_Satisfaction", "Social_Support"
] if c in df_gmm.columns]

perfil_gmm = df_gmm.groupby("cluster_gmm")[perfil_cols].mean().round(2)
perfil_gmm

## 12. Algoritmo 2 — Spectral Clustering

In [ ]:
# ============================================================
# AUMENTO DE CARACTERÍSTICAS PARA SPECTRAL CLUSTERING
# ============================================================

# Se reduce primero con PCA para evitar una explosión dimensional excesiva.
# Luego se generan interacciones cuadráticas con PolynomialFeatures.
# Esto busca capturar relaciones no lineales entre variables.

pca = PCA(n_components=0.90, random_state=SEED)
X_pca = pca.fit_transform(X)

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_pca)
X_poly = StandardScaler().fit_transform(X_poly)

# Para que Spectral Clustering sea más manejable, se usa UMAP de 5 dimensiones sobre las características aumentadas.
X_poly_umap_5d = umap.UMAP(
    n_components=5,
    n_neighbors=20,
    min_dist=0.1,
    random_state=SEED,
).fit_transform(X_poly)

print("Dimensión original:", X.shape)
print("Dimensión PCA:", X_pca.shape)
print("Dimensión luego de PolynomialFeatures:", X_poly.shape)
print("Dimensión UMAP 5D para Spectral:", X_poly_umap_5d.shape)

In [ ]:
# ============================================================
# OPTIMIZACIÓN DE HIPERPARÁMETROS PARA SPECTRAL CLUSTERING
# ============================================================

spectral_resultados = []

param_grid_spectral = {
    "n_clusters": range(2, 9),
    "n_neighbors": [10, 15, 20, 30],
}

for params in ParameterGrid(param_grid_spectral):
    modelo_tmp = SpectralClustering(
        n_clusters=params["n_clusters"],
        affinity="nearest_neighbors",
        n_neighbors=params["n_neighbors"],
        assign_labels="kmeans",
        random_state=SEED,
        n_jobs=-1,
    )
    labels_tmp = modelo_tmp.fit_predict(X_poly_umap_5d)
    m = metricas_internas(X_umap, labels_tmp)
    spectral_resultados.append({
        **params,
        "silhouette": m["silhouette"],
        "davies_bouldin": m["davies_bouldin"],
        "calinski_harabasz": m["calinski_harabasz"],
    })

spectral_resultados = pd.DataFrame(spectral_resultados)
# Se prioriza mayor Silhouette y menor Davies-Bouldin.
spectral_resultados = spectral_resultados.sort_values(
    ["silhouette", "davies_bouldin"], ascending=[False, True]
)

spectral_resultados.head(10)

In [ ]:
# ============================================================
# ENTRENAMIENTO FINAL DE SPECTRAL CLUSTERING
# ============================================================

best_spectral = spectral_resultados.iloc[0]

modelo_spectral = SpectralClustering(
    n_clusters=int(best_spectral["n_clusters"]),
    affinity="nearest_neighbors",
    n_neighbors=int(best_spectral["n_neighbors"]),
    assign_labels="kmeans",
    random_state=SEED,
    n_jobs=-1,
)

labels_spectral = modelo_spectral.fit_predict(X_poly_umap_5d)

print("Mejores hiperparámetros Spectral:")
print(best_spectral[["n_clusters", "n_neighbors", "silhouette", "davies_bouldin"]])

print("\nMétricas Spectral:")
print(evaluar_modelo("Spectral Clustering", labels_spectral))

In [ ]:
# ============================================================
# VISUALIZACIÓN Y PERFIL DE SPECTRAL CLUSTERING
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(X_umap[:, 0], X_umap[:, 1], c=labels_spectral, s=8, alpha=0.7, cmap="tab10")
axes[0].set_title("Spectral Clustering sobre proyección UMAP")
axes[0].set_xlabel("UMAP 1")
axes[0].set_ylabel("UMAP 2")
plt.colorbar(sc, ax=axes[0], label="Cluster")

df_spectral = df_proc.copy()
df_spectral["cluster_spectral"] = labels_spectral
perfil_spectral = df_spectral.groupby("cluster_spectral")[perfil_cols].mean().round(2)
perfil_spectral.T.plot(kind="bar", ax=axes[1])
axes[1].set_title("Perfil promedio por cluster — Spectral")
axes[1].set_ylabel("Promedio")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

perfil_spectral

## 13. Algoritmo 3 — HDBSCAN

In [ ]:
# ============================================================
# OPTIMIZACIÓN DE HIPERPARÁMETROS PARA HDBSCAN
# ============================================================

hdbscan_resultados = []

param_grid_hdbscan = {
    "min_cluster_size": [20, 30, 50, 80],
    "min_samples": [5, 10, 20],
}

for params in ParameterGrid(param_grid_hdbscan):
    modelo_tmp = hdbscan.HDBSCAN(
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
        metric="euclidean",
    )
    labels_tmp = modelo_tmp.fit_predict(X_umap)
    m = metricas_internas(X_umap, labels_tmp)
    hdbscan_resultados.append({
        **params,
        "clusters": contar_clusters(labels_tmp),
        "ruido_%": round(100 * np.mean(labels_tmp == -1), 2),
        "silhouette": m["silhouette"],
        "davies_bouldin": m["davies_bouldin"],
    })

hdbscan_resultados = pd.DataFrame(hdbscan_resultados)

# Se descartan configuraciones con menos de 2 clusters y luego se ordena por Silhouette.
hdbscan_resultados_validos = hdbscan_resultados[hdbscan_resultados["clusters"] >= 2].copy()
hdbscan_resultados_validos = hdbscan_resultados_validos.sort_values(
    ["silhouette", "ruido_%"], ascending=[False, True]
)

hdbscan_resultados_validos.head(10)

In [ ]:
# ============================================================
# ENTRENAMIENTO FINAL DE HDBSCAN
# ============================================================

if len(hdbscan_resultados_validos) == 0:
    raise RuntimeError("Ninguna configuración de HDBSCAN produjo al menos dos clusters.")

best_hdbscan = hdbscan_resultados_validos.iloc[0]

modelo_hdbscan = hdbscan.HDBSCAN(
    min_cluster_size=int(best_hdbscan["min_cluster_size"]),
    min_samples=int(best_hdbscan["min_samples"]),
    metric="euclidean",
)

labels_hdbscan = modelo_hdbscan.fit_predict(X_umap)

print("Mejores hiperparámetros HDBSCAN:")
print(best_hdbscan[["min_cluster_size", "min_samples", "clusters", "ruido_%", "silhouette"]])

print("\nMétricas HDBSCAN:")
print(evaluar_modelo("HDBSCAN", labels_hdbscan))

In [ ]:
# ============================================================
# VISUALIZACIÓN Y PERFIL DE HDBSCAN
# ============================================================

plt.figure(figsize=(8, 6))
plt.scatter(X_umap[:, 0], X_umap[:, 1], c=labels_hdbscan, s=8, alpha=0.7, cmap="tab20")
plt.title("HDBSCAN sobre proyección UMAP")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.colorbar(label="Cluster, -1 = ruido")
plt.tight_layout()
plt.show()

df_hdbscan = df_proc.copy()
df_hdbscan["cluster_hdbscan"] = labels_hdbscan
perfil_hdbscan = df_hdbscan[df_hdbscan["cluster_hdbscan"] != -1].groupby("cluster_hdbscan")[perfil_cols].mean().round(2)
perfil_hdbscan

## 14. Análisis complementario — Isolation Forest para anomalías

In [ ]:
# ============================================================
# DETECCIÓN DE ANOMALÍAS CON ISOLATION FOREST
# ============================================================

modelo_iso = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=SEED,
)

labels_iso = modelo_iso.fit_predict(X)
scores_iso = modelo_iso.decision_function(X)

print("Cantidad de outliers detectados:", int(np.sum(labels_iso == -1)))
print("Porcentaje de outliers:", round(100 * np.mean(labels_iso == -1), 2), "%")

In [ ]:
# ============================================================
# VISUALIZACIÓN DE OUTLIERS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_umap[:, 0], X_umap[:, 1], c=np.where(labels_iso == -1, 1, 0), s=8, alpha=0.7, cmap="coolwarm")
axes[0].set_title("Isolation Forest: outliers sobre UMAP")
axes[0].set_xlabel("UMAP 1")
axes[0].set_ylabel("UMAP 2")

axes[1].hist(scores_iso, bins=40)
axes[1].axvline(0, linestyle="--", color="black")
axes[1].set_title("Distribución del score de anomalía")
axes[1].set_xlabel("Score, mayor = más normal")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

# Perfil promedio de inliers y outliers.
df_iso = df_proc.copy()
df_iso["grupo_iso"] = np.where(labels_iso == -1, "Outlier", "Inlier")
perfil_iso = df_iso.groupby("grupo_iso")[perfil_cols].mean().round(2)
perfil_iso

## 15. Comparación general de modelos

In [ ]:
# ============================================================
# TABLA COMPARATIVA DE MÉTRICAS
# ============================================================

resumen_modelos = pd.DataFrame([
    evaluar_modelo("GMM", labels_gmm),
    evaluar_modelo("Spectral Clustering", labels_spectral),
    evaluar_modelo("HDBSCAN", labels_hdbscan),
])

resumen_modelos = resumen_modelos.round(4)
resumen_modelos

In [ ]:
# ============================================================
# COMPARACIÓN VISUAL DE LOS AGRUPAMIENTOS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

modelos_plot = [
    ("GMM", labels_gmm),
    ("Spectral", labels_spectral),
    ("HDBSCAN", labels_hdbscan),
]

for ax, (nombre, labels) in zip(axes, modelos_plot):
    sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], c=labels, s=8, alpha=0.7, cmap="tab20")
    ax.set_title(nombre)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

plt.suptitle("Comparación de algoritmos de clustering sobre UMAP", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 16. Interpretación crítica de resultados

La interpretación debe realizarse después de ejecutar el notebook, observando las métricas y los perfiles promedio de cada cluster.

Aspectos a considerar:

- **Silhouette:** valores más altos indican clusters más compactos y separados.
- **Davies-Bouldin:** valores más bajos indican mejor separación entre grupos.
- **Calinski-Harabasz:** valores más altos suelen indicar mejor estructura de agrupamiento.
- **ARI y NMI:** si existe una etiqueta externa, estas métricas indican cuánto se parecen los clusters a dicha etiqueta. En un problema no supervisado, no necesariamente se espera una coincidencia alta.
- **HDBSCAN:** puede etiquetar puntos como ruido, lo cual es útil si hay observaciones atípicas.
- **GMM:** permite interpretar incertidumbre de asignación mediante probabilidades.
- **Spectral Clustering:** puede capturar estructuras no lineales, pero depende de la construcción del grafo y de sus hiperparámetros.

Un buen modelo no debe elegirse solo por una métrica. También debe considerarse la interpretabilidad de los perfiles encontrados y la coherencia con el problema de salud mental y estilo de vida.

## 17. Conclusiones

- Se realizó un proceso completo de aprendizaje no supervisado sobre un dataset público de salud mental y estilo de vida.
- El EDA permitió revisar calidad de datos, distribuciones, correlaciones y posibles outliers antes del clustering.
- Se aplicó preprocesamiento mediante imputación, codificación one-hot y escalado estándar, decisiones necesarias para trabajar con algoritmos basados en distancia, densidad o grafos.
- Se implementaron tres algoritmos de clustering distintos a K-Means: GMM, Spectral Clustering y HDBSCAN.
- Se optimizaron hiperparámetros relevantes para cada modelo y se compararon mediante métricas internas.
- La visualización con UMAP permitió observar la estructura de los datos en dos dimensiones sin depender de PCA como técnica principal de visualización.
- Isolation Forest permitió complementar el análisis detectando observaciones atípicas.
- La interpretación final debe basarse en la combinación de métricas, visualizaciones y perfiles de clusters, no únicamente en el valor numérico de una métrica.

En conclusión, el proyecto cumple con el flujo metodológico solicitado: dataset público, EDA, preprocesamiento, clustering con más de dos técnicas, optimización, evaluación, visualización e interpretación crítica.

## 18. Referencias

- Kaggle Dataset: `dhrubangtalukdar/global-mental-health-and-lifestyle-survey-dataset`.
- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python*. Journal of Machine Learning Research.
- McInnes, Healy & Melville (2018). *UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction*.
- Campello, Moulavi & Sander (2013). *Density-Based Clustering Based on Hierarchical Density Estimates*.
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*.

## 19. Checklist final para la entrega

- [x] Dataset público y correctamente referenciado.  
- [x] Descripción de observaciones, variables y fuente.  
- [x] EDA con calidad de datos, estadísticas, distribuciones, correlaciones y outliers.  
- [x] Preprocesamiento justificado.  
- [x] Al menos dos algoritmos de clustering distintos a K-Means.  
- [x] Optimización de hiperparámetros.  
- [x] Métricas internas: Silhouette, Davies-Bouldin y Calinski-Harabasz.  
- [x] Métricas externas si existe etiqueta disponible: ARI y NMI.  
- [x] Visualización con UMAP, técnica distinta de PCA.  
- [x] Discusión crítica, conclusiones y referencias.  
- [x] Código organizado, comentado y reproducible.